# 06 — Chat: grounded answers about ONE opportunity

> **Restart kernel + Run All.** After editing any `.py` file: restart the kernel,
> then Run All (no `%autoreload` — it leaves two copies of the same Pydantic class
> alive and breaks validation).

> **Cost gate:** every answer is **one cached structured model call**, and a real
> provider is called only when `RUN_LIVE_LLM=1` *and* a provider with a key is
> configured. Everywhere else — and in every cell of this notebook except step 8 —
> the offline `fake` provider answers: deterministic, free, zero network. A closed
> gate, a 429 or any model failure gives a **rules-only** answer, never a crash.

**The contract** (`oppradar_ai.chat.answer`, frozen for the backend):

```python
answer(profile, opportunity, history, message, *, settings=None) -> ChatAnswer
```

1. **Grounded.** The model sees only the *public* opportunity record, your question,
   the recent conversation and abstract requirement verdicts. Anything the record
   does not state is answered with the exact phrase **"not in the source"** and
   `in_source=False`.
2. **Private.** Eligibility is computed **locally** by the rules (`check_criteria`).
   The model gets `MET` / `NOT MET` / `UNKNOWN` per requirement — never your CV,
   summary, education, e-mail or the rules' evidence lines.
3. **Pure.** `answer()` stores nothing; the caller (this notebook, the widget)
   persists turns with `JsonStore.append_chat`.
4. **Honest.** Quotes in `citations` are checked against the record; a quote the
   record does not contain is dropped, and the URL is always the record's own.

**Steps:** 1 setup · 2 normal Q&A · 3 grounded refusal · 4 eligibility + what the
model sees · 5 conversation memory · 6 languages · 7 pasted announcement and
links · 8 quota honesty and the gated live example · 9 the chat widget.

In [ ]:
import json
import os
import sys
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

import oppradar_ai.chat as chat_module
from oppradar_ai.chat import REFUSAL_PHRASE, answer, build_prompt
from oppradar_ai.chat_ui import ChatPanel
from oppradar_ai.config import Settings, get_settings, setup_logging
from oppradar_ai.llm import QuotaExhausted, get_llm
from oppradar_ai.matching import check_criteria
from oppradar_ai.schemas import ChatAnswer, ChatTurn, Opportunity, Profile
from oppradar_ai.store import JsonStore

# -- Cost gate: a real model is reachable only when you opt in -----------------
RUN_LIVE_LLM = Settings().run_live_llm
os.environ.pop('LLM_PROVIDER', None)      # drop the force of a previous run
_configured = Settings().llm_provider     # fresh read of ai/.env (get_settings is cached)
_can_live = _configured != 'gemini' or bool(Settings().gemini_key_value)
os.environ['LLM_PROVIDER'] = _configured if RUN_LIVE_LLM and _can_live else 'fake'
get_settings.cache_clear()

setup_logging('INFO')
settings = get_settings().ensure_dirs()
_live = 'yes' if RUN_LIVE_LLM and _can_live else 'no'
print(f'live LLM: {_live} | provider: {settings.llm_provider} | '
      f'history turns sent to the model: {settings.chat_history_turns}')

## Step 1 — A throwaway store, one user, one opportunity

The demo writes to its own store (`data/store_chat_demo/`, wiped on every run)
seeded with the committed fixtures — your real store is never touched. We chat as
`u-amine` (a synthetic profile) about `opp-01`.

In [ ]:
DEMO_STORE = JsonStore(root=settings.data_dir / 'store_chat_demo')
DEMO_STORE.reset()  # a fresh demo on every run; your real store is untouched

fixtures = settings.data_dir / 'fixtures'
profiles = {p['user_id']: Profile.model_validate(p)
            for p in json.loads((fixtures / 'profiles.json').read_text(encoding='utf-8'))}
opps = {o['id']: Opportunity.model_validate(o)
        for o in json.loads((fixtures / 'opportunities.json').read_text(encoding='utf-8'))}
for profile in profiles.values():
    DEMO_STORE.save_user(profile)
DEMO_STORE.save_opportunities(list(opps.values()))

amine, opp = profiles['u-amine'], opps['opp-01']
print(f'demo store: {len(opps)} opportunities x {len(profiles)} profiles')
print(f'chatting as {amine.user_id} about {opp.id}: {opp.title} ({opp.organizer})')


def show(question, reply):
    """Print one exchange: the answer, whether it is grounded, and its sources."""
    print('Q:', question)
    print('A:', reply.answer.replace('\n', '\n   '))
    sources = [f'{c.title} - "{c.quote}"' if c.quote else c.title for c in reply.citations]
    print(f'   in_source={reply.in_source} | language={reply.language} | '
          f'sources: {sources or "none"}')

## Step 2 — Normal Q&A: answered from the record, with a checked citation

The citation's `quote` is a phrase that really appears in the record, and its URL is
always the record's own. `answer()` stored nothing — we check that at the end.

In [ ]:
for question in ('What is the deadline?', 'How much funding is offered?'):
    show(question, answer(amine, opp, [], question, settings=settings))
    print()
print(f'turns stored by answer(): {len(DEMO_STORE.chat_history(amine.user_id))}')

## Step 3 — Grounded refusal: what the source does not say

Ask for something the record never states (an application fee) or a field it leaves
empty (we blank the funding): the reply must contain **"not in the source"**, set
`in_source=False` and point to the page to verify on — it must never guess.

In [ ]:
fee = answer(amine, opp, [], 'What is the application fee?', settings=settings)
show('What is the application fee?', fee)
print(f'refusal phrase present: {REFUSAL_PHRASE in fee.answer.casefold()} | '
      f'in_source={fee.in_source}')
print()

no_funding = opp.model_copy(update={'funding': None})
blank = answer(amine, no_funding, [], 'How much funding is offered?', settings=settings)
show('How much funding is offered? (funding removed from the record)', blank)
print(f'refusal phrase present: {REFUSAL_PHRASE in blank.answer.casefold()} | '
      f'in_source={blank.in_source}')

## Step 4 — Eligibility is computed locally; the model sees only verdicts

`check_criteria` compares each requirement with the profile **on this machine**; its
*evidence* column quotes the profile, so it never leaves it. The reply carries the
local verdict in `ChatAnswer.eligibility`. Below, `build_prompt` prints exactly the
user message the model would receive (the fixed instructions live in
`prompts/answer_question.md`) — and we prove no profile text is in it.

In [ ]:
rows = [{'requirement': c.text, 'status': c.status.value, 'evidence (local only)': c.evidence}
        for c in check_criteria(amine, opp)]
display(pd.DataFrame(rows))

reply = answer(amine, opp, [], 'Am I eligible for this internship?', settings=settings)
show('Am I eligible for this internship?', reply)
verdict = reply.eligibility
print(f'   eligible={verdict.eligible} | met={verdict.met} | gaps={verdict.gaps}')
print('   next steps:', *verdict.next_steps, sep='\n     - ')

In [ ]:
prompt = build_prompt(amine, opp, [], 'Am I eligible for this internship?', settings=settings)
print(prompt)

private = [amine.full_name, amine.email, amine.summary, amine.headline,
           *[e.institution for e in amine.education],
           *[x.organization for x in amine.experience]]
leaked = [text for text in private if text and text.casefold() in prompt.casefold()]
print(f'\nmodel saw the profile: {bool(leaked)} | leaked fields: {leaked}')

## Step 5 — Conversation memory: persisted by the caller, trimmed for the model

`answer()` is pure, so *we* append each turn with `JsonStore.append_chat` and hand
the stored history back on the next question. Only the last
`CHAT_HISTORY_TURNS` turns (default 10) reach the model. Asking the same question
again is a free disk-cache hit.

In [ ]:
def ask(question, opportunity=opp, profile=amine, store=DEMO_STORE):
    """One chat round, the way the widget does it: read history, answer, persist both turns."""
    history = store.chat_history(profile.user_id, limit=settings.chat_history_turns)
    reply = answer(profile, opportunity, history, question, settings=settings)
    store.append_chat(profile.user_id, ChatTurn(role='user', content=question))
    store.append_chat(profile.user_id, ChatTurn(role='assistant', content=reply.answer))
    return reply


for question in ('What is the deadline?', 'And the funding?', 'Where is it located?'):
    show(question, ask(question))
    print()

stored = DEMO_STORE.chat_history(amine.user_id)
print(f'history round trip: {len(stored)} turns stored | roles: {[t.role for t in stored]}')

history_prompt = build_prompt(amine, opp, stored, 'Anything else?', settings=settings)
print('the follow-up prompt carries the conversation:', 'Recent conversation' in history_prompt)

# The same question with the same history is one model call, then a free cache hit.
question = 'What is the deadline?'
cache_prompt = build_prompt(amine, opp, [], question, settings=settings)
answer(amine, opp, [], question, settings=settings)
answer(amine, opp, [], question, settings=settings)
system = (settings.prompts_dir / 'answer_question.md').read_text(encoding='utf-8')
print(f'second ask served from the cache: '
      f'{get_llm().is_cached(cache_prompt, ChatAnswer, system=system)}')

## Step 6 — Languages

The reply follows the language of the question (`ChatAnswer.language`); the offline
fake mimics that from a cheap local hint (`detect_language`), a real model decides
for itself. The refusal phrase stays in English whatever the language.

In [ ]:
for question in ('Quelle est la date limite ?', 'ما هو الموعد النهائي؟', 'ما هي رسوم التسجيل؟'):
    show(question, answer(amine, opp, [], question, settings=settings))
    print()

## Step 7 — A pasted announcement or a link becomes `extracted_opportunity`

A message that carries a link or a long pasted announcement goes through the same
extraction as ingestion (`api.extract_opportunity`: explicit `Deadline:` lines beat
the model, listicles are rejected), and the answer comes from the extracted record.

* **Pasted text** works offline here: the fake model returns a canned record, but
  the rule-first `Deadline:` line is read from *your* text. With a real provider the
  whole record reflects the text.
* **A link** is fetched only behind the gate (`RUN_LIVE_LLM=1` + a real provider,
  robots.txt honoured); offline the reply says so instead of fetching.

In [ ]:
announcement = (
    'Atlas Digital Foundation is opening applications for the Atlas Merit Scholarship.\n'
    'The programme funds one year of postgraduate study in data science for students '
    'from Africa and covers tuition, a monthly stipend and a laptop.\n'
    'Deadline: 2027-02-01\n'
    'Eligibility: open to students; applicants must write English; under 30 years old.\n'
    "Apply on the foundation's website with a motivation letter and two references."
)
reply = answer(amine, None, [], 'What is the deadline?\n\n' + announcement, settings=settings)
extracted = reply.extracted_opportunity
print(f'pasted text extracted: {extracted.title if extracted else None} | '
      f'deadline {extracted.deadline if extracted else None}')
show('What is the deadline? (+ pasted announcement)', reply)
print('   eligibility vs the extracted record:', reply.eligibility.eligible,
      '| gaps:', reply.eligibility.gaps)

In [ ]:
link = answer(amine, opp, [], 'Is this still open? https://careers.acme-example.net/job/42',
              settings=settings)
print('link (offline): extracted =', link.extracted_opportunity)
print('    ' + link.answer.splitlines()[-1])

junk = 'Top 10 scholarships you must not miss! ' + 'Read on for the full list. ' * 20
rejected = answer(amine, opp, [], 'Deadline? ' + junk, settings=settings)
print('listicle rejected -> still answers from the selected opportunity:',
      rejected.extracted_opportunity is None)
print('   ', rejected.answer.splitlines()[-1])

## Step 8 — Quota honesty, then the gated live example

A 429 (or any model failure) is never retried and never crashes the chat: the reply
is a **rules-only** `ChatAnswer` — deadline, funding and the local requirement
verdicts — with a one-line note. The cell below *simulates* a 429 with a stub model
(no network); the live example after it runs only with `RUN_LIVE_LLM=1`.

In [ ]:
class ExhaustedModel:
    """Stands in for a provider whose free quota is gone (a 429)."""
    calls = 0

    def structured(self, prompt, schema, system=None):
        type(self).calls += 1
        raise QuotaExhausted('429 RESOURCE_EXHAUSTED (simulated)')


open_gate = settings.model_copy(update={'llm_provider': 'gemini', 'run_live_llm': True})
real_get_llm = chat_module.get_llm
chat_module.get_llm = lambda provider=None: ExhaustedModel()
try:
    fallback = answer(amine, opp, [], 'What is the deadline?', settings=open_gate)
finally:
    chat_module.get_llm = real_get_llm  # always restore the real factory

show('What is the deadline? (simulated 429)', fallback)
print(f'simulated 429 -> rules-only answer after {ExhaustedModel.calls} model call (no retry)')

closed_gate = settings.model_copy(update={'llm_provider': 'gemini', 'run_live_llm': False})
print('gate closed for a real provider:', answer(amine, opp, [], 'Deadline?',
                                                 settings=closed_gate).answer.split('.')[0] + '.')

In [ ]:
LIVE_URL = ''  # optional: paste an opportunity page here to try the link path live

if settings.llm_provider != 'fake' and RUN_LIVE_LLM:
    question = 'Is this opportunity open to students outside Algeria?'
    show(question, answer(amine, opp, [], question, settings=settings))
    if LIVE_URL:
        live = answer(amine, None, [], f'What is the deadline? {LIVE_URL}', settings=settings)
        show(f'What is the deadline? {LIVE_URL}', live)
else:
    print('skipped (offline): set RUN_LIVE_LLM=1 to ask a real model here')

## Step 9 — The chat widget

`ChatPanel` is a thin ipywidgets shell: a transcript, a text box and a Send button.
It only calls `chat.answer()` and `JsonStore.append_chat()`; it shows the "not in the
source" badge, the sources and the eligibility box, switches to the opportunity found
in a pasted message, and **never raises into the kernel** (a failure is shown in the
status line). Because it reads the same store, it opens with the Step-5 conversation.
Type in the box below (the offline model answers deadline, funding, requirements,
location and benefits questions).

In [ ]:
panel = ChatPanel(amine, opp, DEMO_STORE, settings=settings)
panel.send('What are the benefits?')   # a scripted turn so the transcript is not empty
display(panel.container)
print(f'chat widget: ready | turns now stored: {len(DEMO_STORE.chat_history(amine.user_id))}')